# Bolt #6: Residual Analysis & Reporting

**Objective**: Analyze the Hybrid Baseline residuals to identify performance bottlenecks and validate statistical assumptions.

## 1. Data Loading & Enrichment

Joining OOF residuals with Oil prices and Holiday events.

In [31]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import scipy.stats as stats
from statsmodels.stats.stattools import durbin_watson

# Configuration
RUN_PATH = Path("../../artifacts/runs/test_run/")
RAW_DATA_PATH = Path("../../data/raw/")

print(f"Analyzing run: {RUN_PATH.resolve().name}")

# Load OOF Residuals
df_oof = pd.read_csv(RUN_PATH / "oof_residuals.csv", parse_dates=["date"])

# Load Oil & Holidays
df_oil = pd.read_csv(RAW_DATA_PATH / "oil.csv", parse_dates=["date"])
df_oil = df_oil.set_index("date").resample('D').ffill().reset_index()
df_holidays = pd.read_csv(RAW_DATA_PATH / "holidays_events.csv", parse_dates=["date"])

# Merge
df = df_oof.merge(df_oil, on="date", how="left")
df = df.merge(df_holidays, on="date", how="left")

Analyzing run: test_run


## 2. Segment Metrics & Macro Performance

Identifying high-level failure patterns across the Store-Family matrix.

In [32]:
# Calculate RMSLE per segment
agg_metrics = df.groupby(['store_nbr', 'family']).agg({
    'sq_log_error': 'mean',
    'sales': ['sum', 'mean'],
    'residual': 'std'
}).reset_index()

agg_metrics.columns = ['store_nbr', 'family', 'rmsle_sq', 'total_sales', 'avg_sales', 'residual_std']
agg_metrics['rmsle'] = np.sqrt(agg_metrics['rmsle_sq'])

print(f"Average Segment RMSLE: {agg_metrics['rmsle'].mean():.4f}")

Average Segment RMSLE: 1.6318


In [33]:
# Macro Heatmap
fig_map = px.density_heatmap(
    agg_metrics, x="family", y="store_nbr", z="rmsle",
    title="Global RMSLE Heatmap (Store vs Family)",
    color_continuous_scale="Viridis"
)
fig_map.update_layout(xaxis={'categoryorder':'total descending'})
fig_map.show()

In [34]:
# Boxplots: Error distribution across Groups
fig_store = px.box(agg_metrics, x="store_nbr", y="rmsle", title="RMSLE Distribution per Store")
fig_store.show()

fig_fam = px.box(agg_metrics, x="family", y="rmsle", title="RMSLE Distribution per Family")
fig_fam.update_layout(xaxis={'categoryorder':'median descending'})
fig_fam.show()

## 3. Statistical Residual Analysis

Validating if the residuals follow theoretical assumptions.

In [35]:
# Normality
residuals_sample = df['residual'].dropna().sample(min(5000, len(df)))
_, p_k2 = stats.normaltest(residuals_sample)
print(f"D'Agostino's K^2 p-value: {p_k2:.4e}")

fig_dist = px.histogram(df, x="residual", marginal="box", title="Distribution of Residuals (Pred - Actual)")
fig_dist.show()

D'Agostino's K^2 p-value: 0.0000e+00


In [36]:
# Autocorrelation
dw_stat = durbin_watson(df['residual'].dropna())
print(f"Global Durbin-Watson: {dw_stat:.4f}")
print("Interpretation: ~2.0 is Ideal, <1.5 is Positive Correlation (Missing lags).")

Global Durbin-Watson: 1.7989
Interpretation: ~2.0 is Ideal, <1.5 is Positive Correlation (Missing lags).


## 4. Blind Spot Deep-Dives (Top 5)

Decomposing the error into Trend and Residual contributions (where available).

In [37]:
def plot_segment_deepdive(store_nbr, family):
    subset = df[(df['store_nbr'] == store_nbr) & (df['family'] == family)].sort_values('date')
    
    # Subplots: 1. Sales vs Pred, 2. Components (Log), 3. Residuals
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
        subplot_titles=("Actual vs Predicted Sales", "Hybrid Components (Log-Space)", "Residuals (Error)")
    )
    
    # Row 1: Linear Space Sales
    fig.add_trace(go.Scatter(x=subset['date'], y=subset['sales'], name="Actual", line=dict(color='blue')), row=1, col=1)
    fig.add_trace(go.Scatter(x=subset['date'], y=subset['sales_pred'], name="Predicted", line=dict(color='orange', dash='dash')), row=1, col=1)
    
    # Row 2: Log Space Components (If columns exist)
    if 'trend_pred_log' in subset.columns:
        fig.add_trace(go.Scatter(x=subset['date'], y=subset['trend_pred_log'], name="Trend (Log)", line=dict(color='green')), row=2, col=1)
        fig.add_trace(go.Scatter(x=subset['date'], y=subset['residual_pred_log'], name="Resid Model (Log)", line=dict(color='purple')), row=2, col=1)
    else:
        # Fallback message
        fig.add_annotation(text="Component data missing in this run artifact", row=2, col=1)
        
    # Row 3: Residuals
    fig.add_trace(go.Bar(x=subset['date'], y=subset['residual'], name="Residual", marker_color='red'), row=3, col=1)
    
    # V-Lines for Holidays and Promos
    holiday_dates = subset[subset['type'].notnull()]['date']
    for h_date in holiday_dates:
        fig.add_vline(x=h_date, line_width=1, line_dash="dot", line_color="green", row='all', col=1)
        
    fig.update_layout(height=800, title_text=f"Deep Dive: {store_nbr}-{family}", showlegend=True)
    return fig

top_5 = agg_metrics.sort_values('rmsle', ascending=False).head(5)
for _, row in top_5.iterrows():
    plot_segment_deepdive(row['store_nbr'], row['family']).show()